# [Notebook]Unihack26_Transformer_C2_S02
Research pretraining on audited Parcel3D official train pixels with synthetic appearance masks. Validation selects weights; calibration/test untouched. No physical dent accuracy claim. Source renders remain academic-only. Existing legacy bundles are preserved. A paused session exports resume.pt; do not report it as a completed training run.

In [ ]:
from pathlib import Path
import sys,json,hashlib
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/train/__init__.py': '', 'ml/models/__init__.py': '"""Research model candidates; promotion requires the real-data release gates."""\n', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/data/parcel_pairs_v2.py': '"""Appearance-change pretraining on audited Parcel3D pixels; not physical dent labels."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom ml.data.parcel3d_audit import within_root\n\nMANIFEST_SHA = "267dd46092d3429dc66f539fb897e79b4be3f68beb94b75cd19f7289250e4d38"\nPAIR_VERSION = "parcel-appearance-pairs-v2"\n\n\ndef appearance_pair(rgb, foreground, seed, scenario):\n    """Aligned pair and exact masks for introduced appearance changes only."""\n    rng = np.random.default_rng(seed)\n    h, w = foreground.shape\n    reference = rgb.copy()\n    region = cv2.erode(foreground.astype(np.uint8), np.ones((7, 7), np.uint8))\n    coordinates = np.argwhere(region > 0)\n    if len(coordinates) < 16:\n        raise ValueError("INSUFFICIENT_FOREGROUND")\n    y, x = coordinates[int(rng.integers(len(coordinates)))]\n    change = np.zeros((h, w), np.uint8)\n    if scenario in (1, 3):\n        end = (int(np.clip(x + rng.integers(-w // 6, w // 6 + 1), 0, w - 1)),\n               int(np.clip(y + rng.integers(-h // 6, h // 6 + 1), 0, h - 1)))\n        cv2.line(change, (int(x), int(y)), end, 1, int(rng.integers(2, 6)))\n    elif scenario in (2, 5):\n        radius = max(4, min(h, w) // 20)\n        pts = np.array([x, y]) + rng.integers(-radius, radius + 1, (8, 2))\n        cv2.fillConvexPoly(change, cv2.convexHull(pts.astype(np.int32)), 1)\n    change *= region\n    query = reference.copy()\n    if change.any():\n        color = rng.integers(20, 240, 3).astype(np.float32)\n        alpha = rng.uniform(.45, .95)\n        query[change > 0] = np.clip(query[change > 0] * (1 - alpha) + color * alpha, 0, 255)\n    if scenario == 3:\n        # A pre-existing mark appears in BOTH images. Its target is unchanged.\n        reference = query.copy()\n        change[:] = 0\n    gradient = np.linspace(rng.uniform(.8, 1.2), rng.uniform(.8, 1.2), w)[None, :, None]\n    query = np.clip(query * gradient + rng.uniform(-10, 10), 0, 255).astype(np.uint8)\n    if rng.random() < .3:\n        query = cv2.GaussianBlur(query, (3, 3), .5)\n    valid = np.ones((h, w), np.uint8)\n    if scenario in (4, 5):\n        cx, cy = int(rng.integers(w // 4, w // 2)), int(rng.integers(h // 4, h // 2))\n        ex, ey = min(w - 1, cx + w // 5), min(h - 1, cy + h // 5)\n        query[cy:ey, cx:ex] = rng.integers(20, 240, 3)\n        valid[max(0, cy - 3):min(h, ey + 3), max(0, cx - 3):min(w, ex + 3)] = 0\n    shift = np.float32([[1, 0, rng.uniform(-.6, .6)], [0, 1, rng.uniform(-.6, .6)]])\n    reference = cv2.warpAffine(reference, shift, (w, h), borderMode=cv2.BORDER_REFLECT_101)\n    valid[:3] = valid[-3:] = 0\n    valid[:, :3] = valid[:, -3:] = 0\n    def convert(a):\n        return torch.from_numpy(np.ascontiguousarray(a.transpose(2, 0, 1))).float() / 255\n    return {"query": convert(query), "reference": convert(reference),\n            "target": torch.from_numpy(change[None].astype(np.float32)),\n            "valid": torch.from_numpy(valid[None].astype(np.float32))}\n\n\nclass ParcelPairs(Dataset):\n    def __init__(self, root, manifest, role, size=392, fixture=False):\n        if role not in ("train", "selection", "calibration", "test") or size % 14:\n            raise ValueError("INVALID_ROLE_OR_RESOLUTION")\n        self.root, self.role, self.size, self.epoch = Path(root), role, size, 0\n        raw = Path(manifest).read_bytes()\n        self.manifest_sha256 = hashlib.sha256(raw).hexdigest()\n        if not fixture and self.manifest_sha256 != MANIFEST_SHA:\n            raise ValueError("AUDITED_MANIFEST_HASH_MISMATCH")\n        self.rows = sorted((json.loads(line) for line in raw.splitlines()), key=lambda r: r["image_path"])\n        self.rows = [r for r in self.rows if r["role"] == role]\n        if not self.rows:\n            raise ValueError("EMPTY_ROLE")\n        self.annotations = {}\n        for split in sorted({r["source_split"] for r in self.rows}):\n            document = json.loads((self.root / f"parcel3d_{split}.json").read_text())\n            images = {item["id"]: item["file_name"] for item in document["images"]}\n            for annotation in document["annotations"]:\n                self.annotations[images[annotation["image_id"]]] = annotation\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        raw = within_root(self.root, row["image_path"]).read_bytes()\n        if hashlib.sha256(raw).hexdigest() != row["sha256"]:\n            raise ValueError("IMAGE_CHECKSUM_MISMATCH")\n        import io\n        with Image.open(io.BytesIO(raw)) as source:\n            rgb = np.asarray(source.convert("RGB"))\n        annotation = self.annotations[row["image_path"]]\n        if annotation.get("bbox_mode") != 1 or not isinstance(annotation["segmentation"], list):\n            raise ValueError("UNSUPPORTED_PARCEL_ANNOTATION")\n        foreground = np.zeros(rgb.shape[:2], np.uint8)\n        polygons = [np.array(p, dtype=np.float32).reshape(-1, 2).round().astype(np.int32)\n                    for p in annotation["segmentation"]]\n        cv2.fillPoly(foreground, polygons, 1)\n        x, y, w, h = annotation["bbox"]\n        pad = int(max(w, h) * .12)\n        x0, y0 = max(0, int(x) - pad), max(0, int(y) - pad)\n        x1, y1 = min(rgb.shape[1], int(x + w) + pad), min(rgb.shape[0], int(y + h) + pad)\n        rgb = cv2.resize(rgb[y0:y1, x0:x1], (self.size, self.size))\n        foreground = cv2.resize(foreground[y0:y1, x0:x1], (self.size, self.size), interpolation=cv2.INTER_NEAREST)\n        epoch = self.epoch if self.role == "train" else 0\n        identity = int(hashlib.sha256(row["image_path"].encode()).hexdigest()[:8], 16)\n        scenario = (index + epoch) % 6\n        pair = appearance_pair(rgb, foreground, [26, identity, epoch], scenario)\n        return {**pair, "source": row["image_path"], "lineage": row["lineage_id"], "scenario": scenario}\n', 'ml/models/paired_transformer.py': '"""Spatial paired-image candidate, not a validated damage detector.\n\nBoth images must already be in the query coordinate system. Matching/visibility\nis an upstream task. Unknown pixels are excluded from change supervision.\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nDINO_REVISION = "7764ea0f912e53c92e82eb78a2a1631e92725fc8"\nBACKBONES = {\n    "dinov2_vitb14": {"channels": 768, "layers": (2, 5, 8, 11)},\n    "dinov2_vitl14": {"channels": 1024, "layers": (5, 11, 17, 23)},\n}\n\n\nclass LoRALinear(nn.Module):\n    """Low-rank adaptation of a frozen attention projection."""\n\n    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 16):\n        super().__init__()\n        if rank < 1:\n            raise ValueError("LORA_RANK_MUST_BE_POSITIVE")\n        self.base = base.requires_grad_(False)\n        self.down = nn.Linear(base.in_features, rank, bias=False)\n        self.up = nn.Linear(rank, base.out_features, bias=False)\n        self.scale = alpha / rank\n        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))\n        nn.init.zeros_(self.up.weight)\n\n    def forward(self, x):\n        return self.base(x) + self.up(self.down(x)) * self.scale\n\n\nclass DinoSpatialEncoder(nn.Module):\n    def __init__(self, name="dinov2_vitl14", lora_blocks=0, cache_dir=None):\n        super().__init__()\n        if name not in BACKBONES or not 0 <= lora_blocks <= 4:\n            raise ValueError("UNSUPPORTED_ENCODER_CONFIGURATION")\n        if cache_dir:\n            torch.hub.set_dir(str(cache_dir))\n        self.backbone = torch.hub.load(\n            f"facebookresearch/dinov2:{DINO_REVISION}", name,\n            pretrained=True, trust_repo=True,\n        ).requires_grad_(False)\n        self.name = name\n        self.channels = BACKBONES[name]["channels"]\n        self.layers = BACKBONES[name]["layers"]\n        self.adapted = lora_blocks > 0\n        if self.adapted:\n            for block in self.backbone.blocks[-lora_blocks:]:\n                block.attn.qkv = LoRALinear(block.attn.qkv)\n        self.register_buffer("mean", torch.tensor([.485, .456, .406])[None, :, None, None])\n        self.register_buffer("std", torch.tensor([.229, .224, .225])[None, :, None, None])\n\n    def forward(self, images):\n        if images.shape[-1] % 14 or images.shape[-2] % 14:\n            raise ValueError("ENCODER_SIZE_MUST_BE_MULTIPLE_OF_14")\n        # No stochastic depth in the frozen foundation model. LoRA still gets\n        # gradients in training; no inference-mode tensors enter the decoder.\n        self.backbone.eval()\n        gradients = torch.is_grad_enabled() and self.training and self.adapted\n        with torch.set_grad_enabled(gradients):\n            return list(self.backbone.get_intermediate_layers(\n                (images - self.mean) / self.std, n=self.layers, reshape=True,\n            ))\n\n\nclass CrossViewBlock(nn.Module):\n    """Query patches attend to the aligned reference\'s spatial features."""\n\n    def __init__(self, width=128, heads=4):\n        super().__init__()\n        self.q_norm = nn.LayerNorm(width)\n        self.r_norm = nn.LayerNorm(width)\n        self.attention = nn.MultiheadAttention(width, heads, batch_first=True)\n        self.position = nn.Conv2d(2, width, 1)\n        self.fuse = nn.Sequential(\n            nn.Conv2d(width * 3, width, 1), nn.GroupNorm(8, width), nn.GELU(),\n            nn.Conv2d(width, width, 3, padding=1), nn.GroupNorm(8, width), nn.GELU(),\n        )\n\n    def forward(self, query, reference, valid):\n        b, _, h, w = query.shape\n        yy, xx = torch.meshgrid(\n            torch.linspace(-1, 1, h, device=query.device, dtype=query.dtype),\n            torch.linspace(-1, 1, w, device=query.device, dtype=query.dtype), indexing="ij",\n        )\n        position = self.position(torch.stack([xx, yy])[None]).flatten(2).transpose(1, 2)\n        q = self.q_norm(query.flatten(2).transpose(1, 2))\n        r = self.r_norm(reference.flatten(2).transpose(1, 2))\n        keep = F.interpolate(valid.float(), (h, w), mode="area").flatten(1) >= .5\n        empty = ~keep.any(dim=1)\n        # PyTorch attention is undefined if every key is masked. A dummy key is\n        # enabled only for the numerical operation; its output is then zeroed.\n        safe_keep = keep.clone()\n        safe_keep[empty, 0] = True\n        context, _ = self.attention(\n            q + position, r + position, r,\n            key_padding_mask=~safe_keep, need_weights=False,\n        )\n        context = context.masked_fill(empty[:, None, None], 0)\n        context = context.transpose(1, 2).reshape(b, -1, h, w)\n        return self.fuse(torch.cat([query, query - reference, query - context], 1))\n\n\nclass PairedChangeTransformer(nn.Module):\n    def __init__(self, encoder_channels=1024, width=128, levels=4):\n        super().__init__()\n        if width % 8 or levels < 1:\n            raise ValueError("INVALID_DECODER_CONFIGURATION")\n        self.levels = levels\n        self.projections = nn.ModuleList(nn.Conv2d(encoder_channels, width, 1) for _ in range(levels))\n        self.cross_view = nn.ModuleList(CrossViewBlock(width) for _ in range(levels))\n        self.semantic = nn.Sequential(nn.Conv2d(levels * width, width, 1), nn.GELU())\n        # RGB detail bypass preserves narrow scratches that patch tokens lose.\n        self.detail = nn.Sequential(\n            nn.Conv2d(10, 32, 5, stride=2, padding=2), nn.GroupNorm(8, 32), nn.GELU(),\n            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.GroupNorm(8, 32), nn.GELU(),\n        )\n        self.decode = nn.Sequential(\n            nn.Conv2d(width + 32, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),\n            nn.Conv2d(64, 32, 3, padding=1), nn.GELU(), nn.Conv2d(32, 1, 1),\n        )\n\n    def forward(self, query_features, reference_features, query, reference, valid):\n        if len(query_features) != self.levels or len(reference_features) != self.levels:\n            raise ValueError("FEATURE_LEVEL_COUNT_MISMATCH")\n        shape = query_features[0].shape[-2:]\n        fused = []\n        for q, r, projection, block in zip(\n            query_features, reference_features, self.projections, self.cross_view, strict=True,\n        ):\n            item = block(projection(q), projection(r), valid)\n            fused.append(F.interpolate(item, shape, mode="bilinear", align_corners=False))\n        semantic = self.semantic(torch.cat(fused, 1))\n        detail = self.detail(torch.cat([query, reference, (query - reference).abs(), valid], 1))\n        semantic = F.interpolate(semantic, detail.shape[-2:], mode="bilinear", align_corners=False)\n        logits = self.decode(torch.cat([semantic, detail], 1))\n        return F.interpolate(logits, query.shape[-2:], mode="bilinear", align_corners=False)\n\n\ndef masked_change_loss(logits, target, valid):\n    """Unknown pixels contribute neither gradients nor positive/negative labels."""\n    # Half-precision sums overflow above 65,504 pixels at the target resolutions.\n    logits, target, valid = logits.float(), target.float(), valid.float()\n    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")\n    bce = (bce * valid).sum() / valid.sum().clamp_min(1)\n    p, t = logits.sigmoid() * valid, target * valid\n    dims = (1, 2, 3)\n    dice = 1 - (2 * (p * t).sum(dims) + 1) / (p.sum(dims) + t.sum(dims) + 1)\n    observed = valid.sum(dims) > 0\n    return bce + (dice * observed).sum() / observed.sum().clamp_min(1)\n', 'ml/train/paired_transformer.py': '"""Resumable appearance pretraining, selection-only evaluation, no test tuning."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Subset\n\nfrom ml.data.parcel3d_audit import find_annotations\nfrom ml.data.parcel_pairs_v2 import PAIR_VERSION, ParcelPairs\nfrom ml.models.paired_transformer import (\n    DINO_REVISION,\n    DinoSpatialEncoder,\n    PairedChangeTransformer,\n    masked_change_loss,\n)\n\n\ndef trainable_state(encoder, decoder):\n    return {"encoder": {name: p.detach().cpu().clone() for name, p in encoder.named_parameters() if p.requires_grad},\n            "decoder": {name: value.detach().cpu().clone() for name, value in decoder.state_dict().items()}}\n\n\ndef load_trainable(state, encoder, decoder):\n    expected = {name: p for name, p in encoder.named_parameters() if p.requires_grad}\n    if set(expected) != set(state["encoder"]):\n        raise ValueError("ADAPTER_PARAMETER_SET_MISMATCH")\n    with torch.no_grad():\n        for name, value in state["encoder"].items():\n            if expected[name].shape != value.shape:\n                raise ValueError("ADAPTER_PARAMETER_SHAPE_MISMATCH")\n            expected[name].copy_(value)\n    decoder.load_state_dict(state["decoder"], strict=True)\n\n\ndef atomic_save(value, path):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef forward_pair(encoder, decoder, batch, device):\n    q, r, y, valid = [batch[key].to(device) for key in ("query", "reference", "target", "valid")]\n    with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):\n        n = len(q)\n        features = encoder(torch.cat([q, r]))\n        logits = decoder([x[:n] for x in features], [x[n:] for x in features], q, r, valid)\n        value = masked_change_loss(logits, y, valid)\n    return value\n\n\ndef run(input_root, manifest, output, config, profile, resume=None, fixture=False, clock=None):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    now = clock or time.monotonic\n    started = now()\n    device = torch.device("cuda" if torch.cuda.is_available() and not fixture else "cpu")\n    if not fixture:\n        if device.type != "cuda":\n            raise ValueError("TRAINING_REQUIRES_GPU")\n        matching = [r for r in profile.get("measurements", []) if r.get("status") == "PASS"\n                    and r["encoder"] == config["encoder"] and r["size"] == config["size"]\n                    and r["lora_blocks"] == config["lora_blocks"]]\n        if not matching:\n            raise ValueError("EXACT_CONFIGURATION_PROFILE_REQUIRED")\n        backbone_sha = matching[0]["backbone_sha256"]\n    else:\n        backbone_sha = "SOFTWARE_FIXTURE_NO_PRETRAINED_WEIGHTS"\n    torch.manual_seed(config["seed"])\n    torch.set_num_threads(2)\n    root = find_annotations(Path(input_root))["train"].parent\n    train = ParcelPairs(root, manifest, "train", config["size"], fixture=fixture)\n    selection = ParcelPairs(root, manifest, "selection", config["size"], fixture=fixture)\n    encoder = DinoSpatialEncoder(config["encoder"], config["lora_blocks"], "/kaggle/temp/kc-model-cache").to(device)\n    if not fixture:\n        weights = Path(torch.hub.get_dir()) / "checkpoints" / f"{config[\'encoder\']}_pretrain.pth"\n        digest = hashlib.sha256()\n        with weights.open("rb") as stream:\n            for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n                digest.update(chunk)\n        if digest.hexdigest() != backbone_sha:\n            raise ValueError("BACKBONE_PROFILE_HASH_MISMATCH")\n    decoder = PairedChangeTransformer(encoder.channels, width=config["width"]).to(device)\n    adapters = [p for p in encoder.parameters() if p.requires_grad]\n    parameters = list(decoder.parameters()) + adapters\n    groups = [{"params": list(decoder.parameters()), "lr": config["head_lr"]}]\n    if adapters:\n        groups.append({"params": adapters, "lr": config["adapter_lr"]})\n    optimizer = torch.optim.AdamW(groups, weight_decay=.01)\n    total_updates = math.ceil(len(train) / config["accumulation_steps"]) * config["epochs"]\n    warmup = min(200, max(1, total_updates // 10))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda step: min(1., (step + 1) / warmup) *\n        (.1 + .45 * (1 + math.cos(math.pi * min(step, total_updates) / total_updates))),\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    source_root = Path(__file__).resolve().parents[2]\n    source_hashes = {name: hashlib.sha256((source_root / name).read_bytes()).hexdigest() for name in\n                     ("ml/models/paired_transformer.py", "ml/data/parcel_pairs_v2.py", "ml/train/paired_transformer.py")}\n    identity = {"config": {k: v for k, v in config.items() if k != "max_seconds"}, "source_hashes": source_hashes,\n                "manifest_sha256": train.manifest_sha256, "pair_version": PAIR_VERSION,\n                "backbone_revision": DINO_REVISION, "backbone_sha256": backbone_sha,\n                "source_kind": "SOFTWARE_FIXTURE" if fixture else "SYNTHETIC_CHANGES_ON_PARCEL3D_RENDERS"}\n    epoch, offset, updates, attempts, skipped, best = 0, 0, 0, 0, 0, None\n    history = []\n    if resume:\n        state = torch.load(resume, map_location="cpu", weights_only=True)\n        if state["identity"] != identity:\n            raise ValueError("RESUME_CONFIGURATION_OR_DATA_CHANGED")\n        load_trainable(state["last"], encoder, decoder)\n        optimizer.load_state_dict(state["optimizer"])\n        scheduler.load_state_dict(state["scheduler"])\n        scaler.load_state_dict(state["scaler"])\n        epoch, offset, updates, best, history = [state[k] for k in ("epoch", "offset", "updates", "best", "history")]\n        attempts, skipped = state["attempts"], state["skipped"]\n        torch.set_rng_state(state["rng_cpu"])\n        if device.type == "cuda":\n            torch.cuda.set_rng_state_all(state["rng_cuda"])\n\n    def checkpoint(status):\n        atomic_save({"identity": identity, "last": trainable_state(encoder, decoder),\n                     "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n                     "epoch": epoch, "offset": offset, "updates": updates, "best": best,\n                     "attempts": attempts, "skipped": skipped,\n                     "history": history, "rng_cpu": torch.get_rng_state(),\n                     "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else []}, output / "resume.pt")\n        card = {"status": status, **identity, "epochs_completed": epoch, "next_sample_offset": offset,\n                "optimizer_updates": updates, "train_source_images": len(train),\n                "optimizer_attempts": attempts, "skipped_amp_steps": skipped,\n                "selection_source_images": len(selection), "selection_loss": None if best is None else best["loss"],\n                "seconds_this_session": now() - started, "test_evaluation": "NOT_RUN",\n                "real_box_validation": "NOT_RUN", "promotion": "HOLD",\n                "limitations": ["Paired labels are synthetic appearance changes, not 3D dent truth.",\n                                "Inputs use known alignment and foreground; upstream matching is not evaluated.",\n                                "No evidence of outperforming the legacy model on real parcels."]}\n        (output / "model-card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")\n        (output / "training-log.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        print(json.dumps(card), flush=True)\n        return card\n\n    accumulation = config["accumulation_steps"]\n    for current_epoch in range(epoch, config["epochs"]):\n        epoch = current_epoch\n        train.epoch = epoch\n        encoder.train()\n        decoder.train()\n        order = np.random.default_rng([config["seed"], epoch]).permutation(len(train)).tolist()\n        loader = DataLoader(Subset(train, order[offset:]), batch_size=1, shuffle=False, num_workers=0)\n        iterator = iter(loader)\n        while offset < len(train):\n            # Check only at optimizer boundaries so no uncommitted gradients or\n            # consumed samples disappear when a session resumes.\n            if now() - started >= config["max_seconds"]:\n                return checkpoint("PAUSED_SESSION_BUDGET")\n            count = min(accumulation, len(train) - offset)\n            optimizer.zero_grad(set_to_none=True)\n            total = 0.\n            for _ in range(count):\n                value = forward_pair(encoder, decoder, next(iterator), device)\n                if not torch.isfinite(value):\n                    checkpoint("FAILED_NONFINITE_LOSS")\n                    raise ValueError("NONFINITE_TRAINING_LOSS")\n                scaler.scale(value / count).backward()\n                total += float(value.detach())\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(parameters, 1.)\n            scale_before = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            offset += count\n            attempts += 1\n            if scaler.get_scale() < scale_before:\n                skipped += 1\n            else:\n                updates += 1\n                scheduler.step()\n            if attempts % 100 == 0:\n                history.append({"epoch": epoch, "offset": offset, "update": updates, "last_batch_loss": total / count})\n                checkpoint("RUNNING")\n        # Selection is deterministic and never touches calibration/test images.\n        encoder.eval()\n        decoder.eval()\n        losses = []\n        with torch.no_grad():\n            for batch in DataLoader(selection, batch_size=1, num_workers=0):\n                value = forward_pair(encoder, decoder, batch, device)\n                if not torch.isfinite(value):\n                    raise ValueError("NONFINITE_SELECTION_LOSS")\n                losses.append(float(value))\n        score = float(np.mean(losses))\n        history.append({"epoch": epoch + 1, "selection_loss": score, "update": updates})\n        if best is None or score < best["loss"]:\n            best = {"loss": score, "epoch": epoch + 1, "weights": trainable_state(encoder, decoder)}\n            atomic_save({"identity": identity, **best}, output / "best.pt")\n        epoch, offset = epoch + 1, 0\n        checkpoint("EPOCH_COMPLETE")\n    if best is not None:\n        # Also export the best checkpoint carried from a previous session.\n        atomic_save({"identity": identity, **best}, output / "best.pt")\n    return checkpoint("PRETRAINING_COMPLETE_REQUIRES_CALIBRATION_AND_REAL_VALIDATION")\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/train/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/models/__init__.py': '970a22e9a2f8997752da0cb86ff07508de56946ba32d03e4d2d9b92a35f29399', 'ml/data/parcel3d_audit.py': '08ddf5f683f5b23c8364228d53f8b3883518465c3e2d66a64bf3ff9888f46b4e', 'ml/data/parcel_pairs_v2.py': '26d32013957f1bc2cc281bcf4b8d8a4ba15a702899125313b58b2a5d11f30730', 'ml/models/paired_transformer.py': 'd211fafb77b78b555fc9863c85178962dce3b1a39851343b82befb63663154e1', 'ml/train/paired_transformer.py': '512ef3246575cebd7dcbcf735b44c863ed09c68f8ab20beafcdc0c7fdee32a3f'},indent=2))


In [ ]:
CONFIG={'experiment': 'parcel-transformer-c2-v1', 'encoder': 'dinov2_vitl14', 'lora_blocks': 0, 'width': 128, 'size': 392, 'seed': 26, 'epochs': 30, 'accumulation_steps': 8, 'head_lr': 0.0001, 'adapter_lr': 1e-05, 'max_seconds': 14400}
PROFILE={'purpose': 'FEASIBILITY_ONLY_NOT_TRAINING_OR_ACCURACY', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda_available': True, 'measurements': [{'encoder': 'dinov2_vitb14', 'size': 392, 'lora_blocks': 0, 'batch_pairs': 1, 'trainable_parameters': 1644929, 'encoder_parameters': 86580480, 'status': 'PASS', 'mean_step_seconds': 0.06783407974999989, 'p95_step_seconds': 0.08135331100000229, 'peak_allocated_bytes': 501559296, 'peak_reserved_bytes': 562036736, 'backbone_sha256': '0b8b82f85de91b424aded121c7e1dcc2b7bc6d0adeea651bf73a13307fad8c73', 'finite_losses': [1.4785243272781372, 1.4266777038574219, 1.387791395187378, 1.3558626174926758], 'note': 'Single repeated synthetic pair; losses are NOT a benchmark.'}, {'encoder': 'dinov2_vitl14', 'size': 392, 'lora_blocks': 0, 'batch_pairs': 1, 'trainable_parameters': 1776001, 'encoder_parameters': 304368640, 'status': 'PASS', 'mean_step_seconds': 0.10366481274999018, 'p95_step_seconds': 0.10734076039999109, 'peak_allocated_bytes': 1379929088, 'peak_reserved_bytes': 1444937728, 'backbone_sha256': 'd5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428', 'finite_losses': [1.5449604988098145, 1.5227441787719727, 1.503591537475586, 1.486230492591858], 'note': 'Single repeated synthetic pair; losses are NOT a benchmark.'}, {'encoder': 'dinov2_vitl14', 'size': 392, 'lora_blocks': 4, 'batch_pairs': 1, 'trainable_parameters': 1907073, 'encoder_parameters': 304499712, 'status': 'PASS', 'mean_step_seconds': 0.1184501489999974, 'p95_step_seconds': 0.1187184916499973, 'peak_allocated_bytes': 1643479040, 'peak_reserved_bytes': 1721761792, 'backbone_sha256': 'd5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428', 'finite_losses': [1.5516633987426758, 1.5155110359191895, 1.4851157665252686, 1.4598032236099243], 'note': 'Single repeated synthetic pair; losses are NOT a benchmark.'}, {'encoder': 'dinov2_vitl14', 'size': 518, 'lora_blocks': 0, 'batch_pairs': 1, 'trainable_parameters': 1776001, 'encoder_parameters': 304368640, 'status': 'PASS', 'mean_step_seconds': 0.1597647289999884, 'p95_step_seconds': 0.1604790258499932, 'peak_allocated_bytes': 1460438528, 'peak_reserved_bytes': 1547698176, 'backbone_sha256': 'd5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428', 'finite_losses': [1.434791922569275, 1.3918545246124268, 1.3574413061141968, 1.3295347690582275], 'note': 'Single repeated synthetic pair; losses are NOT a benchmark.'}], 'gpu': 'Tesla T4', 'gpu_total_bytes': 15636037632, 'source_image_sha256': 'a9eae709843a4aeefc8817b54a3c79e3fb69ca54b8a45c80a19bccd5d543aaa5', 'source_split': 'OFFICIAL_TRAIN_ONLY', 'elapsed_seconds': 49.446308345999995, 'status': 'COMPLETE'}
RESUME_SHA='5cb200b96411dec285c94ebcdb9d9f6d8fa6e1a38161f7331767df9972d71ab4'
from collections import deque
from ml.data.parcel_pairs_v2 import MANIFEST_SHA
from ml.train.paired_transformer import run
queue=deque([(Path('/kaggle/input'),0)])
found=set()
resumes=set()
while queue:
    directory,depth=queue.popleft()
    path=directory/'images.jsonl'
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        found.add(path.resolve())
    resume_path=directory/'resume.pt'
    if RESUME_SHA and resume_path.is_file() and hashlib.sha256(resume_path.read_bytes()).hexdigest()==RESUME_SHA:
        resumes.add(resume_path.resolve())
    if depth<6:
        for child in directory.iterdir():
            if child.is_dir() and child.name not in {'parcel3d','train','validation','test','misc'}:
                queue.append((child,depth+1))
assert found,'AUDITED_IMAGE_MANIFEST_INPUT_MISSING'
assert not RESUME_SHA or resumes,'VERIFIED_RESUME_INPUT_MISSING'
result=run('/kaggle/input',sorted(found)[0],'/kaggle/working/training',CONFIG,PROFILE,
           resume=sorted(resumes)[0] if resumes else None)
